In [ ]:
# # ======================= explainability_no_batching.py =======================
# # Paste this entire cell into 06_explainability.ipynb and run.
# # It processes chunks one-by-one (no padding) so it avoids all previous batching errors.
# # Start with SAMPLE_LIMIT_PER_LANG = 2 for a smoke-run, then set None for full run.

# import os, json, random, time, html
# from pathlib import Path
# from typing import List, Dict
# from tqdm import tqdm

# import numpy as np
# import pandas as pd
# import torch
# from transformers import AutoTokenizer, AutoModelForSequenceClassification

# # Optional Captum
# CAPTUM = False
# try:
#     from captum.attr import IntegratedGradients
#     CAPTUM = True
# except Exception:
#     CAPTUM = False

# # ---------- CONFIG ----------
# ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2")
# DATA_SPLITS_ROOT = ROOT / "data_splits"
# DISTILL_OUT = ROOT / "outputs" / "distillation"
# EXPLAIN_OUT_ROOT = ROOT / "outputs" / "explainability"
# STUDENT_MODEL_ID = "distil_distilbert-base-multilingual-cased"
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# SAMPLE_LIMIT_PER_LANG = 2   # set to None to process all files
# DELETION_FRACS = np.linspace(0.0, 0.5, 11)
# TOP_TOKENS_TO_SAVE = 30
# BATCH_SIZE = 1  # unused; kept for readability
# SEED = 42
# random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
# if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# # ---------- helpers ----------
# def ensure_dir(p: Path):
#     p.mkdir(parents=True, exist_ok=True); return p

# def read_label_map(lang_dir: Path):
#     p = lang_dir / "label_map.json"
#     if not p.exists(): raise FileNotFoundError(f"label_map.json not found: {p}")
#     jd = json.load(open(p, encoding="utf8"))
#     label_map = {str(k): int(v) for k,v in jd.get("label_map", {}).items()}
#     inv_map = {int(v): str(k) for k,v in jd.get("label_map", {}).items()}
#     metadata = jd.get("metadata", {})
#     return label_map, inv_map, metadata

# def softmax_np(x: np.ndarray, axis=-1):
#     e = np.exp(x - np.max(x, axis=axis, keepdims=True))
#     return e / (e.sum(axis=axis, keepdims=True) + 1e-12)

# # ------------------ REPLACE chunk_texts_by_tokenizer WITH THIS ------------------
# def chunk_texts_by_tokenizer(tokenizer, text: str, max_len:int, stride:int):
#     """
#     Safe chunker: uses tokenizer(...) WITHOUT return_tensors.
#     Then converts each overflowed chunk individually to a tensor of shape (1, seq_len).
#     This avoids tokenizer trying to stack lists of different lengths into a single tensor.
#     """
#     text = "" if text is None else str(text)
#     # ask for python lists (no return_tensors)
#     enc = tokenizer(
#         text,
#         truncation=True,
#         return_overflowing_tokens=True,
#         max_length=max_len,
#         stride=stride,
#         # do NOT pass return_tensors here
#         # optionally request offsets if your code uses them
#         return_offsets_mapping=False
#     )

#     # enc contains lists in enc['input_ids'] etc. Each element is a list (per chunk).
#     chunks = []
#     # number of chunks:
#     n_chunks = len(enc['input_ids']) if isinstance(enc['input_ids'], list) else (enc['input_ids'].shape[0] if hasattr(enc['input_ids'], 'shape') else 0)

#     # iterate safely and create a per-chunk tensor (shape (1, seq_len))
#     for i in range(n_chunks):
#         ids = enc['input_ids'][i]
#         att = enc['attention_mask'][i] if 'attention_mask' in enc else [1]*len(ids)
#         # sometimes tokenizers give numpy arrays -> convert to list then tensor
#         if hasattr(ids, "tolist"):
#             try:
#                 ids = ids.tolist()
#             except Exception:
#                 ids = list(ids)
#         if hasattr(att, "tolist"):
#             try:
#                 att = att.tolist()
#             except Exception:
#                 att = list(att)
#         # now convert to torch tensor per chunk
#         ids_tensor = torch.tensor([ids], dtype=torch.long)       # shape (1, seq_len_for_this_chunk)
#         att_tensor = torch.tensor([att], dtype=torch.long)
#         chunks.append({
#             "input_ids": ids_tensor,
#             "attention_mask": att_tensor
#         })
#     return chunks
# # -------------------------------------------------------------------------------


# def file_predict_from_chunks_one_by_one(model, tokenizer, chunks, device):
#     """
#     Run each chunk individually through model (no batch padding).
#     Returns aggregated logits (mean over chunk logits) and predicted label index.
#     """
#     model.eval()
#     logits_list=[]
#     with torch.no_grad():
#         for ch in chunks:
#             ids = ch['input_ids']
#             att = ch['attention_mask']
#             # ensure tensors and shape (1, seq_len)
#             if not torch.is_tensor(ids):
#                 ids = torch.tensor(ids, dtype=torch.long)
#             if ids.dim() == 1:
#                 ids = ids.unsqueeze(0)
#             if not torch.is_tensor(att):
#                 att = torch.tensor(att, dtype=torch.long)
#             if att.dim() == 1:
#                 att = att.unsqueeze(0)
#             ids = ids.to(device); att = att.to(device)
#             out = model(input_ids=ids, attention_mask=att)
#             logits_list.append(out.logits.detach().cpu().numpy())
#     if len(logits_list) == 0:
#         return np.zeros((model.config.num_labels,), dtype=float), 0
#     all_logits = np.vstack(logits_list)
#     agg = all_logits.mean(axis=0)
#     return agg, int(np.argmax(agg))

# def compute_attention_scores_chunkwise(model, tokenizer, chunks, device):
#     """
#     For each chunk, compute average attention across heads & layers and return per-chunk token scores.
#     """
#     model.eval()
#     token_lists=[]; score_lists=[]
#     for ch in chunks:
#         try:
#             ids = ch['input_ids']
#             att = ch['attention_mask']
#             if not torch.is_tensor(ids): ids = torch.tensor(ids)
#             if ids.dim() == 2 and ids.size(0) == 1:
#                 ids_proc = ids.to(device)
#             else:
#                 ids_proc = ids.unsqueeze(0).to(device)
#             if not torch.is_tensor(att): att = torch.tensor(att)
#             att_proc = att.to(device) if att.dim()==2 and att.size(0)==1 else att.unsqueeze(0).to(device)
#             with torch.no_grad():
#                 out = model(input_ids=ids_proc, attention_mask=att_proc, output_attentions=True)
#                 atts = out.attentions  # tuple layers (batch, heads, seq_len, seq_len)
#                 # average over heads and layers
#                 att_mats = [a.mean(dim=1).squeeze(0).detach().cpu().numpy() for a in atts]
#                 mean_att = np.mean(np.stack(att_mats, axis=0), axis=0)  # shape (seq_len, seq_len)
#                 token_scores = mean_att.sum(axis=0).tolist()  # sum over source tokens -> per token importance
#                 token_ids = ids_proc.detach().cpu().squeeze(0).tolist()
#                 toks = tokenizer.convert_ids_to_tokens(token_ids)
#                 token_lists.append(toks); score_lists.append(token_scores)
#         except Exception:
#             token_lists.append([]); score_lists.append([])
#     return token_lists, score_lists

# def compute_gradxinput_chunkwise(model, tokenizer, chunks, device, target_label=None):
#     model.eval()
#     emb = model.get_input_embeddings()
#     token_lists=[]; score_lists=[]
#     for ch in chunks:
#         try:
#             ids = ch['input_ids']
#             att = ch['attention_mask']
#             if not torch.is_tensor(ids): ids = torch.tensor(ids)
#             if ids.dim() == 2 and ids.size(0) == 1:
#                 ids_proc = ids.to(device)
#             else:
#                 ids_proc = ids.unsqueeze(0).to(device)
#             if not torch.is_tensor(att): att = torch.tensor(att)
#             att_proc = att.to(device) if att.dim()==2 and att.size(0)==1 else att.unsqueeze(0).to(device)
#             token_ids = ids_proc.detach().cpu().squeeze(0).tolist()
#             toks = tokenizer.convert_ids_to_tokens(token_ids)
#             if CAPTUM:
#                 try:
#                     ig = IntegratedGradients(model)
#                     baseline = torch.zeros_like(emb(ids_proc))
#                     attributions, _ = ig.attribute(inputs=emb(ids_proc), baselines=baseline, target=target_label, return_convergence_delta=True)
#                     at_sum = attributions.detach().cpu().numpy().sum(axis=2).squeeze(0)
#                     token_lists.append(toks); score_lists.append(at_sum.tolist()); continue
#                 except Exception:
#                     pass
#             # fallback grad*input
#             inp_emb = emb(ids_proc)
#             inp_emb.requires_grad_(True)
#             out = model(inputs_embeds=inp_emb, attention_mask=att_proc)
#             logits = out.logits
#             tgt = int(logits.argmax(dim=-1).item()) if target_label is None else int(target_label)
#             score = logits[0, tgt]
#             model.zero_grad(); score.backward(retain_graph=False)
#             grads = inp_emb.grad.detach().cpu().numpy().squeeze(0)
#             emb_np = inp_emb.detach().cpu().numpy().squeeze(0)
#             gx = (grads * emb_np).sum(axis=1)
#             token_lists.append(toks); score_lists.append(gx.tolist())
#         except Exception:
#             token_lists.append([]); score_lists.append([])
#     return token_lists, score_lists

# def aggregate_tokens_and_scores(token_lists, score_lists):
#     full_tokens=[]; full_scores=[]
#     for toks, sc in zip(token_lists, score_lists):
#         if not toks: continue
#         full_tokens.extend(toks); full_scores.extend([float(x) for x in sc])
#     token_agg={}
#     for t,s in zip(full_tokens, full_scores):
#         token_agg.setdefault(t, []).append(float(s))
#     token_agg_mean = {t: float(np.mean(v)) for t,v in token_agg.items()}
#     return full_tokens, full_scores, token_agg_mean

# def tokens_to_html(token_list, token_scores, title="", outpath:Path=None):
#     if len(token_scores)==0:
#         html_text = "<p>(no tokens)</p>"
#     else:
#         arr = np.array(token_scores, dtype=float)
#         mx = max(abs(arr.min()), abs(arr.max()), 1e-12)
#         norm = arr / mx
#         spans=[]
#         for tok, v in zip(token_list, norm):
#             vv = float(max(-1.0,min(1.0,v)))
#             if vv >= 0:
#                 r=255; g=int(255-(vv*200)); b=int(255-(vv*200))
#             else:
#                 vv2=-vv; r=int(255-(vv2*200)); g=int(255-(vv2*200)); b=255
#             color=f"rgb({r},{g},{b})"
#             token_html = html.escape(tok).replace(" ", "&nbsp;")
#             spans.append(f"<span style='background:{color};padding:0.06rem;margin:0.02rem;border-radius:0.12rem'>{token_html}</span>")
#         html_text = " ".join(spans)
#     full = f"<html><head><meta charset='utf8'><title>{html.escape(title)}</title></head><body><h3>{html.escape(title)}</h3><div style='line-height:1.6;font-family:monospace'>{html_text}</div></body></html>"
#     if outpath: open(outpath, "w", encoding="utf8").write(full)
#     return full

# def deletion_test_simple(model, tokenizer, full_tokens, full_scores, chunks, delete_frac, device):
#     if not full_tokens: return {"delete_frac": delete_frac, "new_prob": 0.0}
#     n=len(full_tokens); k=max(1,int(n*delete_frac))
#     order = np.argsort(-np.array(full_scores)) if len(full_scores)>0 else np.arange(n)
#     remove_set=set(order[:k])
#     kept = [t for i,t in enumerate(full_tokens) if i not in remove_set]
#     s = " ".join(kept).replace(" ##","").replace("Ġ"," ")
#     new_chunks = chunk_texts_by_tokenizer(tokenizer, s, max_len=chunks[0]['input_ids'].size(1), stride=max(32,chunks[0]['input_ids'].size(1)//4)) if chunks else []
#     new_logits, _ = file_predict_from_chunks_one_by_one(model, tokenizer, new_chunks, device)
#     new_probs = softmax_np(new_logits[np.newaxis,:], axis=1)[0] if new_logits is not None else np.array([0.0])
#     orig_logits, orig_label = file_predict_from_chunks_one_by_one(model, tokenizer, chunks, device)
#     orig_probs = softmax_np(orig_logits[np.newaxis,:], axis=1)[0] if orig_logits is not None else np.array([0.0])
#     return {"delete_frac": delete_frac, "orig_label": int(orig_label), "orig_prob": float(orig_probs[orig_label]) if orig_label < len(orig_probs) else 0.0, "new_prob": float(new_probs[orig_label]) if orig_label < len(new_probs) else 0.0}

# # ---------- MAIN ----------
# LANGS = sorted([d.name for d in DATA_SPLITS_ROOT.iterdir() if d.is_dir()])
# print("Languages detected:", LANGS)
# summary_report={}

# for LANG in LANGS:
#     print("\n" + "="*80)
#     print("[LANG]", LANG)
#     lang_dir = DATA_SPLITS_ROOT / LANG
#     label_map, inv_label_map, metadata = read_label_map(lang_dir)
#     chunk_max_len = int(metadata.get("chunk_max_len", 256))
#     chunk_stride = int(metadata.get("chunk_stride", 64))
#     out_dir = ensure_dir(EXPLAIN_OUT_ROOT / LANG / STUDENT_MODEL_ID)
#     html_dir = ensure_dir(out_dir / "html_highlights")
#     student_best = DISTILL_OUT / LANG / STUDENT_MODEL_ID / "best_model"
#     if not student_best.exists():
#         raise FileNotFoundError(f"Missing model dir: {student_best}")
#     print("[INFO] Loading model from:", student_best)
#     tokenizer = AutoTokenizer.from_pretrained(str(student_best), use_fast=True)
#     model = AutoModelForSequenceClassification.from_pretrained(str(student_best)).to(DEVICE)
#     model.eval()

#     df_test = pd.read_csv(lang_dir / "test.csv")
#     rows = list(df_test.itertuples(index=False, name=None))
#     if SAMPLE_LIMIT_PER_LANG:
#         rows = rows[:SAMPLE_LIMIT_PER_LANG]

#     metrics_rows=[]; top_tokens_rows=[]; deletion_rows=[]
#     for idx,row in enumerate(tqdm(rows, desc=f"{LANG} files")):
#         try:
#             cols = list(df_test.columns)
#             rp = {c: getattr(row, i) if isinstance(i,str) else getattr(row, idx_col) for idx_col, c in enumerate(cols)}
#         except Exception:
#             rp = {}
#             for i,c in enumerate(cols):
#                 try: rp[c] = row[i]
#                 except Exception: rp[c] = None
#         fp = rp.get("file_path", "")
#         if not fp:
#             metrics_rows.append({"file_path": fp}); print("[WARN] missing file_path"); continue
#         try:
#             text = Path(fp).read_text(encoding="utf8", errors="ignore")
#         except Exception:
#             text = ""
#         if not text.strip():
#             metrics_rows.append({"file_path": fp}); print(f"[WARN] empty file {fp}"); continue

#         # chunk
#         chunks = chunk_texts_by_tokenizer(tokenizer, text, max_len=chunk_max_len, stride=chunk_stride)
#         if not chunks:
#             enc = tokenizer(text, truncation=True, max_length=chunk_max_len, return_tensors="pt")
#             chunks = [{"input_ids": enc['input_ids'].unsqueeze(0), "attention_mask": enc['attention_mask'].unsqueeze(0)}]

#         # prediction (chunk-by-chunk)
#         agg_logits, pred_label = file_predict_from_chunks_one_by_one(model, tokenizer, chunks, DEVICE)
#         probs = softmax_np(agg_logits[np.newaxis,:], axis=1)[0] if agg_logits is not None else np.array([0.0])
#         pred_prob = float(probs[pred_label]) if pred_label < len(probs) else 0.0

#         # token attributions (attention + gradxinput) per chunk
#         att_tokens, att_scores = compute_attention_scores_chunkwise(model, tokenizer, chunks, DEVICE)
#         grad_tokens, grad_scores = compute_gradxinput_chunkwise(model, tokenizer, chunks, DEVICE, target_label=None)

#         full_tokens, full_scores, composite = aggregate_tokens_and_scores(att_tokens, att_scores)
#         # if composite empty try grad
#         if not composite and grad_tokens:
#             full_tokens, full_scores, composite = aggregate_tokens_and_scores(grad_tokens, grad_scores)

#         # build token CSV
#         token_rows=[]
#         for tok, score in composite.items():
#             token_rows.append({"file_path": fp, "token": tok, "composite_score": float(score)})
#         try:
#             pd.DataFrame(token_rows).to_csv(out_dir / f"tokens_{Path(fp).stem}.csv", index=False, encoding="utf8")
#         except Exception:
#             pass

#         # html highlight
#         try:
#             toks_flat = full_tokens if full_tokens else tokenizer.convert_ids_to_tokens(chunks[0]['input_ids'].squeeze(0).tolist())
#             scores_flat = full_scores if full_scores else [0.0]*len(toks_flat)
#             htmlp = html_dir / f"highlight_{Path(fp).stem}.html"
#             tokens_to_html(toks_flat, scores_flat, title=f"{LANG} - {Path(fp).stem}", outpath=htmlp)
#         except Exception:
#             pass

#         # deletion faithfulness curve (simple)
#         fracs = DELETION_FRACS.tolist()
#         retained_probs=[]
#         for f in fracs:
#             res = deletion_test_simple(model, tokenizer, full_tokens, full_scores, chunks, delete_frac=f, device=DEVICE)
#             retained_probs.append(float(res.get("new_prob", 0.0)))
#             deletion_rows.append({"file_path": fp, "delete_frac": float(f), "retained_prob": float(res.get("new_prob", 0.0))})

#         # top tokens
#         ordered = sorted(composite.items(), key=lambda x:-x[1])[:TOP_TOKENS_TO_SAVE]
#         top_tokens_rows.append({"file_path": fp, "top_tokens": ordered, "n_tokens": len(full_tokens)})

#         # metrics per file
#         metrics_rows.append({
#             "file_path": fp,
#             "pred_label": int(pred_label),
#             "pred_label_str": None,
#             "pred_prob": pred_prob,
#             "n_tokens": len(full_tokens),
#             "deletion_auc": float(np.trapz(retained_probs, fracs)) if len(fracs)>1 else 0.0
#         })
#         print(f"[OK] {LANG} {Path(fp).stem} pred={pred_label} prob={pred_prob:.3f} tokens={len(full_tokens)}")

#     # save artifacts
#     ensure_dir(out_dir)
#     pd.DataFrame(metrics_rows).to_csv(out_dir / "explainability_metrics_per_file.csv", index=False, encoding="utf8")
#     pd.DataFrame(top_tokens_rows).to_csv(out_dir / "top_tokens_per_file.csv", index=False, encoding="utf8")
#     pd.DataFrame(deletion_rows).to_csv(out_dir / "deletion_faithfulness.csv", index=False, encoding="utf8")

#     summary = {"language": LANG, "n_files": len(metrics_rows)}
#     json.dump(summary, open(out_dir / f"explainability_summary_{LANG}.json", "w"), indent=2)
#     summary_report[LANG] = {"out_dir": str(out_dir), "metrics_csv": str(out_dir/"explainability_metrics_per_file.csv"), "htmls": len(list((out_dir/"html_highlights").glob("*.html")))}
#     print(f"[DONE] {LANG} saved to {out_dir}")

# print("\nSMOKE REPORT")
# for k,v in summary_report.items():
#     print(k, v)
# print("\nALL DONE")


In [ ]:
# ================== 06_explainability_batched_full.py ==================
# Full explainability (batched + safe + selective gradient attribution)
# Paste this whole cell into 06_explainability.ipynb and run.
# Start with SAMPLE_LIMIT_PER_LANG = 2 for quick smoke-run.

import os, json, time, html, math, random
from pathlib import Path
from typing import List, Dict
from tqdm import tqdm

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.cuda.amp import autocast

# Optional Captum (if installed)
CAPTUM = False
try:
    from captum.attr import IntegratedGradients
    CAPTUM = True
except Exception:
    CAPTUM = False

# ---------------- CONFIG ----------------
ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2")
DATA_SPLITS_ROOT = ROOT / "data_splits"
DISTILL_OUT = ROOT / "outputs" / "distillation"
EXPLAIN_OUT_ROOT = ROOT / "outputs" / "explainability"
STUDENT_MODEL_ID = "distil_distilbert-base-multilingual-cased"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tunable speed/quality knobs
SAMPLE_LIMIT_PER_LANG = None        # set None for full dataset run
BATCH_SIZE = 16                  # chunk forward batch size
USE_AMP = torch.cuda.is_available()
TOP_K_CHUNKS = 64                # number of top chunks to run gradient attribution on
TOP_TOKENS_TO_SAVE = 30
DELETION_FRACS = np.array([0.0, 0.05, 0.15, 0.3, 0.5])  # coarse curve (faster)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# ---------------- helpers ----------------
def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def read_label_map(lang_dir: Path):
    p = lang_dir / "label_map.json"
    if not p.exists():
        raise FileNotFoundError(f"Missing label_map.json at {p}")
    jd = json.load(open(p, encoding="utf8"))
    label_map = {str(k): int(v) for k,v in jd.get("label_map", {}).items()}
    inv_map = {int(v): str(k) for k,v in jd.get("label_map", {}).items()}
    metadata = jd.get("metadata", {})
    return label_map, inv_map, metadata

def softmax_np(x: np.ndarray, axis=-1):
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / (e.sum(axis=axis, keepdims=True) + 1e-12)

# Safe chunking: tokenizer(..., return_overflowing_tokens=True) without return_tensors
def chunk_texts_safe(tokenizer, text: str, max_len:int, stride:int):
    """
    Safe chunker: uses tokenizer(...) WITHOUT return_tensors.
    Returns list of dicts: {"input_ids": list[int], "attention_mask": list[int]}
    """
    text = "" if text is None else str(text)
    enc = tokenizer(
        text,
        truncation=True,
        return_overflowing_tokens=True,
        max_length=max_len,
        stride=stride,
        return_offsets_mapping=False
    )

    # enc['input_ids'] is a list-like where each item is a list of ids for that chunk
    chunks = []
    if isinstance(enc['input_ids'], (list, tuple)):
        n_chunks = len(enc['input_ids'])
    else:
        # Some tokenizers may return numpy arrays
        n_chunks = enc['input_ids'].shape[0] if hasattr(enc['input_ids'], 'shape') else 0

    for i in range(n_chunks):
        ids = enc['input_ids'][i]
        att = enc['attention_mask'][i] if 'attention_mask' in enc else [1] * len(ids)
        # ensure python lists (avoid numpy arrays)
        if hasattr(ids, "tolist"):
            try:
                ids = ids.tolist()
            except Exception:
                ids = list(ids)
        if hasattr(att, "tolist"):
            try:
                att = att.tolist()
            except Exception:
                att = list(att)
        chunks.append({"input_ids": ids, "attention_mask": att})
    return chunks


# Batched prediction with safe per-batch pad
def predict_chunks_batched(model, tokenizer, chunk_dicts: List[Dict], device, batch_size=BATCH_SIZE, use_amp=USE_AMP):
    model.eval()
    all_logits=[]
    for i in range(0, len(chunk_dicts), batch_size):
        batch_feats = chunk_dicts[i:i+batch_size]
        try:
            batch_t = tokenizer.pad(batch_feats, padding=True, return_tensors="pt")
            input_ids = batch_t["input_ids"].to(device)
            attention_mask = batch_t["attention_mask"].to(device)
        except Exception:
            # fallback manual pad (defensive)
            maxlen = max(len(x["input_ids"]) for x in batch_feats)
            pad_id = tokenizer.pad_token_id or 0
            ids_p=[]; masks_p=[]
            for x in batch_feats:
                ids = x["input_ids"] + [pad_id]*(maxlen-len(x["input_ids"]))
                mask = x["attention_mask"] + [0]*(maxlen-len(x["attention_mask"]))
                ids_p.append(ids); masks_p.append(mask)
            input_ids = torch.tensor(ids_p, dtype=torch.long).to(device)
            attention_mask = torch.tensor(masks_p, dtype=torch.long).to(device)
        with torch.no_grad():
            if use_amp:
                with torch.amp.autocast("cuda"):
                    out = model(input_ids=input_ids, attention_mask=attention_mask, output_attentions=False)
            else:
                out = model(input_ids=input_ids, attention_mask=attention_mask, output_attentions=False)
            all_logits.append(out.logits.detach().cpu().numpy())
    if not all_logits:
        return np.zeros((0, model.config.num_labels))
    return np.vstack(all_logits)

# Batched attention extraction
def attention_scores_batched(model, tokenizer, chunk_dicts: List[Dict], device, batch_size=BATCH_SIZE, use_amp=USE_AMP):
    model.eval()
    token_lists=[]; score_lists=[]
    for i in range(0, len(chunk_dicts), batch_size):
        batch_feats = chunk_dicts[i:i+batch_size]
        try:
            batch_t = tokenizer.pad(batch_feats, padding=True, return_tensors="pt")
            input_ids = batch_t["input_ids"].to(device)
            attention_mask = batch_t["attention_mask"].to(device)
        except Exception:
            maxlen = max(len(x["input_ids"]) for x in batch_feats)
            pad_id = tokenizer.pad_token_id or 0
            ids_p=[]; masks_p=[]
            for x in batch_feats:
                ids = x["input_ids"] + [pad_id]*(maxlen-len(x["input_ids"]))
                mask = x["attention_mask"] + [0]*(maxlen-len(x["attention_mask"]))
                ids_p.append(ids); masks_p.append(mask)
            input_ids = torch.tensor(ids_p, dtype=torch.long).to(device)
            attention_mask = torch.tensor(masks_p, dtype=torch.long).to(device)

        with torch.no_grad():
            if use_amp:
                with torch.amp.autocast("cuda"):
                    out = model(input_ids=input_ids, attention_mask=attention_mask, output_attentions=True)
            else:
                out = model(input_ids=input_ids, attention_mask=attention_mask, output_attentions=True)
        atts = out.attentions
        # atts: tuple(len_layers) of (batch, heads, seq, seq)
        att_per_layer = [a.mean(dim=1).detach().cpu().numpy() for a in atts]
        stacked = np.mean(np.stack(att_per_layer, axis=0), axis=0)  # (batch, seq, seq)
        token_scores_batch = stacked.sum(axis=1)  # (batch, seq)
        input_ids_cpu = input_ids.detach().cpu().numpy()
        mask_cpu = attention_mask.detach().cpu().numpy()
        for b in range(input_ids_cpu.shape[0]):
            ids_b = input_ids_cpu[b].tolist()
            mask_b = mask_cpu[b].tolist()
            valid_len = int(sum(mask_b))
            toks = tokenizer.convert_ids_to_tokens(ids_b)[:valid_len]
            scores = token_scores_batch[b][:valid_len].tolist()
            token_lists.append(toks); score_lists.append(scores)
    return token_lists, score_lists

# chunk-level aggregation & token-level composite
def aggregate_chunk_token_scores(chunk_token_lists, chunk_score_lists):
    full_tokens=[]; full_scores=[]
    for toks, sc in zip(chunk_token_lists, chunk_score_lists):
        if not toks: continue
        full_tokens.extend(toks); full_scores.extend([float(x) for x in sc])
    token_map={}
    for t,s in zip(full_tokens, full_scores):
        token_map.setdefault(t, []).append(float(s))
    token_agg = {t: float(np.mean(v)) for t,v in token_map.items()}
    ordered = sorted(token_agg.items(), key=lambda x:-x[1])
    return full_tokens, full_scores, token_agg, ordered

def select_topk_chunks_by_confidence(chunk_logits: np.ndarray, k=TOP_K_CHUNKS):
    if chunk_logits is None or len(chunk_logits)==0: return []
    probs = softmax_np(chunk_logits, axis=1)
    conf = probs.max(axis=1)
    return np.argsort(-conf)[:k].tolist()

# Grad*input for selected chunks only
def gradxinput_on_selected_chunks(model, tokenizer, chunk_dicts, selected_indices, device, target_label=None, use_captum=CAPTUM):
    results={}
    emb_layer = model.get_input_embeddings()
    for idx in selected_indices:
        if idx < 0 or idx >= len(chunk_dicts): continue
        ch = chunk_dicts[idx]
        ids = ch["input_ids"]
        mask = ch["attention_mask"]
        ids_t = torch.tensor([ids], dtype=torch.long).to(device)
        mask_t = torch.tensor([mask], dtype=torch.long).to(device)
        try:
            inp_emb = emb_layer(ids_t)
            inp_emb = inp_emb.clone().detach().requires_grad_(True)
            out = model(inputs_embeds=inp_emb, attention_mask=mask_t)
            logits = out.logits
            tgt = int(logits.argmax(dim=-1).item()) if target_label is None else int(target_label)
            score = logits[0, tgt]
            model.zero_grad(); score.backward()
            grads = inp_emb.grad.detach().cpu().numpy().squeeze(0)
            emb_np = inp_emb.detach().cpu().numpy().squeeze(0)
            gx = (grads * emb_np).sum(axis=1)
            toks = tokenizer.convert_ids_to_tokens(ids)
            results[idx] = (toks, gx.tolist())
        except Exception as e:
            results[idx] = ([], [])
        # free memory
        try:
            del inp_emb, grads, emb_np, gx
            torch.cuda.empty_cache()
        except Exception:
            pass
    return results

def tokens_to_html(token_list, token_scores, title="", outpath:Path=None):
    if len(token_scores)==0:
        html_text = "<p>(no tokens)</p>"
    else:
        arr = np.array(token_scores, dtype=float)
        mx = max(abs(arr.min()), abs(arr.max()), 1e-12)
        norm = arr / mx
        spans=[]
        for tok, v in zip(token_list, norm):
            vv = float(max(-1.0,min(1.0,v)))
            if vv >= 0:
                r=255; g=int(255-(vv*200)); b=int(255-(vv*200))
            else:
                vv2=-vv; r=int(255-(vv2*200)); g=int(255-(vv2*200)); b=255
            color=f"rgb({r},{g},{b})"
            token_html = html.escape(tok).replace(" ", "&nbsp;")
            spans.append(f"<span style='background:{color};padding:0.06rem;margin:0.02rem;border-radius:0.12rem'>{token_html}</span>")
        html_text = " ".join(spans)
    full = f"<html><head><meta charset='utf8'><title>{html.escape(title)}</title></head><body><h3>{html.escape(title)}</h3><div style='line-height:1.6;font-family:monospace'>{html_text}</div></body></html>"
    if outpath:
        open(outpath, "w", encoding="utf8").write(full)
    return full

def deletion_test_from_ordered(model, tokenizer, ordered_tokens, target_label, fracs, device):
    retained_probs=[]
    for f in fracs:
        k = max(1, int(len(ordered_tokens) * f))
        kept = ordered_tokens[k:]
        s = " ".join(kept).replace(" ##","").replace("Ġ"," ")
        chunks_new = chunk_texts_safe(tokenizer, s, max_len=256, stride=64)
        if not chunks_new:
            enc = tokenizer(s, truncation=True, max_length=256, return_tensors="pt")
            chunks_new = [{"input_ids": enc['input_ids'].unsqueeze(0), "attention_mask": enc['attention_mask'].unsqueeze(0)}]
        logits_new = predict_chunks_batched(model, tokenizer, chunks_new, device, batch_size=BATCH_SIZE, use_amp=USE_AMP)
        if logits_new.size == 0:
            retained_probs.append(0.0)
            continue
        probs_new = softmax_np(logits_new.mean(axis=0, keepdims=True), axis=1)[0]
        retained_probs.append(float(probs_new[target_label]) if target_label < len(probs_new) else 0.0)
    try:
        auc = float(np.trapz(retained_probs, fracs))
    except Exception:
        auc = float(np.trapz(retained_probs, fracs)) if len(fracs)>1 else 0.0
    return retained_probs, auc

# ---------------- main loop ----------------
LANGS = sorted([d.name for d in DATA_SPLITS_ROOT.iterdir() if d.is_dir()])
print("Languages detected:", LANGS)
smoke_report={}

for LANG in LANGS:
    t0 = time.time()
    print("\n" + "="*80)
    print("[LANG]", LANG)
    lang_dir = DATA_SPLITS_ROOT / LANG
    label_map, inv_label_map, metadata = read_label_map(lang_dir)
    chunk_max_len = int(metadata.get("chunk_max_len", 256))
    chunk_stride = int(metadata.get("chunk_stride", 64))

    out_dir = ensure_dir(EXPLAIN_OUT_ROOT / LANG / STUDENT_MODEL_ID)
    html_dir = ensure_dir(out_dir / "html_highlights")

    student_best = DISTILL_OUT / LANG / STUDENT_MODEL_ID / "best_model"
    if not student_best.exists():
        raise FileNotFoundError(f"Missing student best model dir: {student_best}")
    print("[INFO] Loading model from:", student_best)
    tokenizer = AutoTokenizer.from_pretrained(str(student_best), use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(str(student_best)).to(DEVICE)
    model.eval()

    df_test = pd.read_csv(lang_dir / "test.csv")
    rows = list(df_test.itertuples(index=False, name=None))
    if SAMPLE_LIMIT_PER_LANG:
        rows = rows[:SAMPLE_LIMIT_PER_LANG]

    metrics_rows=[]; deletion_rows=[]; top_tokens_rows=[]
    for r in tqdm(rows, desc=f"{LANG} files"):
        # robust row -> dict
        cols = list(df_test.columns)
        try:
            rowdict = {c: getattr(r, i) if isinstance(i,str) else getattr(r, idx) for idx,c in enumerate(cols)}
        except Exception:
            rowdict = {}
            for i,c in enumerate(cols):
                try: rowdict[c]=r[i]
                except Exception: rowdict[c]=None

        fp = rowdict.get("file_path", "")
        if not fp:
            metrics_rows.append({"file_path": fp}); print("[WARN] missing file_path"); continue
        try:
            text = Path(fp).read_text(encoding="utf8", errors="ignore")
        except Exception:
            text = ""

        if not text or text.strip()=="":
            metrics_rows.append({"file_path": fp}); print(f"[WARN] empty file: {fp}"); continue

        # chunking (safe)
        chunks = chunk_texts_safe(tokenizer, text, max_len=chunk_max_len, stride=chunk_stride)
        if not chunks:
            enc = tokenizer(text, truncation=True, max_length=chunk_max_len, return_tensors="pt")
            chunks = [{"input_ids": enc['input_ids'].unsqueeze(0).tolist()[0], "attention_mask": enc['attention_mask'].unsqueeze(0).tolist()[0]}]

        # chunk-level logits (batched)
        chunk_logits = predict_chunks_batched(model, tokenizer, chunks, DEVICE, batch_size=BATCH_SIZE, use_amp=USE_AMP)  # (n_chunks, num_labels)
        if chunk_logits.size == 0:
            metrics_rows.append({"file_path": fp}); print(f"[WARN] no logits for {fp}"); continue

        # file-level aggregate
        agg_logits = chunk_logits.mean(axis=0)
        pred_label = int(np.argmax(agg_logits))
        pred_prob = float(softmax_np(agg_logits[np.newaxis,:], axis=1)[0][pred_label])

        # batched attention extraction
        chunk_token_lists, chunk_att_scores = attention_scores_batched(model, tokenizer, chunks, DEVICE, batch_size=BATCH_SIZE, use_amp=USE_AMP)

        # aggregate token-level composites (attention for all chunks)
        full_tokens, full_scores, token_agg_map, ordered_tokens = aggregate_chunk_token_scores(chunk_token_lists, chunk_att_scores)

        # select top-k chunks by chunk confidence to run gradients on (if any)
        top_chunk_indices = select_topk_chunks_by_confidence(chunk_logits, k=TOP_K_CHUNKS)
        grad_results = gradxinput_on_selected_chunks(model, tokenizer, chunks, top_chunk_indices, DEVICE, target_label=None)

        # ---------- Safe merge of grad results into token_agg_map ----------
# token_agg_map currently maps token -> float (attention mean) OR token -> list (if already merged)
# grad_results maps chunk_idx -> (tokens_list, grad_scores_list)

        for idx, (toks, scores) in grad_results.items():
            # toks, scores can be empty lists on error
            for tok, sc in zip(toks, scores):
                sc_f = float(sc)
                if tok in token_agg_map:
                    existing = token_agg_map[tok]
                    # if existing is a list, append; if float, convert to list
                    if isinstance(existing, list):
                        existing.append(sc_f)
                    else:
                        # convert float -> list [existing, new]
                        token_agg_map[tok] = [float(existing), sc_f]
                else:
                    # create new list entry
                    token_agg_map[tok] = [sc_f]

        # Now convert all entries to a single float by averaging
        composite_map = {t: float(np.mean(v)) if isinstance(v, (list,tuple,np.ndarray)) else float(v)
                        for t, v in token_agg_map.items()}
        ordered_composite = sorted(composite_map.items(), key=lambda x: -x[1])
        ordered_tokens_list = [t for t,_ in ordered_composite]


        # produce token csv
        token_rows=[]
        for tok, val in composite_map.items():
            token_rows.append({"file_path": fp, "token": tok, "composite_score": float(val)})
        try:
            pd.DataFrame(token_rows).to_csv(out_dir / f"tokens_{Path(fp).stem}.csv", index=False, encoding="utf8")
        except Exception:
            pass

        # html highlight
        try:
            toks_for_html = ordered_tokens_list if ordered_tokens_list else full_tokens
            scores_for_html = [composite_map.get(t,0.0) for t in toks_for_html]
            tokens_to_html(toks_for_html, scores_for_html, title=f"{LANG} - {Path(fp).stem}", outpath=html_dir / f"highlight_{Path(fp).stem}.html")
        except Exception:
            pass

        # deletion faithfulness on ordered tokens (fast: uses ordered top tokens)
        retained_probs, del_auc = deletion_test_from_ordered(model, tokenizer, ordered_tokens_list, pred_label, DELETION_FRACS, DEVICE)

        # save per-frac rows
        for frac, rp in zip(DELETION_FRACS.tolist(), retained_probs):
            deletion_rows.append({"file_path": fp, "delete_frac": float(frac), "retained_prob": float(rp)})

        # top tokens summary
        topk = ordered_composite[:TOP_TOKENS_TO_SAVE]
        top_tokens_rows.append({"file_path": fp, "top_tokens": topk, "n_tokens": len(ordered_tokens_list)})

        # metric row (include pred_label_str fix here)
        metrics_rows.append({
            "file_path": fp,
            "pred_label": int(pred_label),
            "pred_label_str": inv_label_map.get(int(pred_label), None),
            "pred_prob": float(pred_prob),
            "n_chunks": int(chunk_logits.shape[0]),
            "n_tokens": int(len(full_tokens)),
            "deletion_auc": float(del_auc)
        })

        print(f"[OK] {LANG} {Path(fp).stem} pred={pred_label} prob={pred_prob:.3f} chunks={chunk_logits.shape[0]} tokens={len(full_tokens)} del_auc={del_auc:.4f}")

    # Save artifacts
    ensure_dir(out_dir)
    metrics_csv = out_dir / "explainability_metrics_per_file.csv"
    metrics_df = pd.DataFrame(metrics_rows)
    # ensure pred_label_str exists (double-check)
    if "pred_label" in metrics_df.columns and "pred_label_str" not in metrics_df.columns:
        try:
            metrics_df["pred_label_str"] = metrics_df["pred_label"].apply(lambda x: inv_label_map.get(int(x)) if pd.notna(x) else None)
        except Exception:
            metrics_df["pred_label_str"] = None
    metrics_df.to_csv(metrics_csv, index=False, encoding="utf8")

    pd.DataFrame(top_tokens_rows).to_csv(out_dir / "top_tokens_per_file.csv", index=False, encoding="utf8")
    pd.DataFrame(deletion_rows).to_csv(out_dir / "deletion_faithfulness.csv", index=False, encoding="utf8")

    # produce per-file deletion summary (one row per file)
    try:
        df_del = pd.read_csv(out_dir / "deletion_faithfulness.csv")
        summary_rows=[]
        for fp, g in df_del.groupby("file_path"):
            fracs = g["delete_frac"].values
            probs = g["retained_prob"].values
            try:
                auc = float(np.trapz(probs, fracs))
            except Exception:
                auc = float("nan")
            summary_rows.append({"file_path": fp, "deletion_auc": auc, "n_points": len(probs)})
        pd.DataFrame(summary_rows).to_csv(out_dir / "deletion_faithfulness_summary.csv", index=False, encoding="utf8")
    except Exception as e:
        print("[WARN] could not write deletion summary:", e)

    # small summary json
    summary = {"language": LANG, "n_files_explained": len(metrics_rows)}
    json.dump(summary, open(out_dir / f"explainability_summary_{LANG}.json", "w"), indent=2)
    smoke_report[LANG] = {"out_dir": str(out_dir), "metrics_csv": str(metrics_csv), "htmls": len(list((out_dir/"html_highlights").glob("*.html")))}
    t1 = time.time()
    print(f"[DONE] {LANG} elapsed_sec={(t1-t0):.1f}")

# final smoke report
print("\nSMOKE REPORT")
for k,v in smoke_report.items():
    print(k, v)
print("\nALL DONE")
# ================= end of cell ==================


Languages detected: ['English', 'Hindi', 'Marathi']

[LANG] English
[INFO] Loading model from: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/distillation/English/distil_distilbert-base-multilingual-cased/best_model


English files:   0%|          | 0/172 [00:00<?, ?it/s]You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
DistilBertSdpaAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.
/tmp/ipython-input-681446518.py:274: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc = float(np.trapz(retained_probs, fracs))
English files:   1%|          | 1/172 [00:09<25:40,  9.01s/it]

[OK] English PG-13_Awakenings_1990 pred=3 prob=0.995 chunks=281 tokens=71787 del_auc=0.0016


English files:   1%|          | 2/172 [00:11<14:57,  5.28s/it]

[OK] English R_Alone_in_the_Dark_2005 pred=2 prob=0.760 chunks=219 tokens=55916 del_auc=0.4993


...

[OK] Marathi U_Khel_Mandala_2012 pred=0 prob=0.515 chunks=100 tokens=25502 del_auc=0.2684


Marathi files: 100%|██████████| 16/16 [00:32<00:00,  2.05s/it]

[OK] Marathi UA_Ringa_Ringa_2010 pred=0 prob=0.509 chunks=64 tokens=16365 del_auc=0.2680



/tmp/ipython-input-681446518.py:449: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc = float(np.trapz(probs, fracs))


[DONE] Marathi elapsed_sec=54.5

SMOKE REPORT
English {'out_dir': '/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/explainability/English/distil_distilbert-base-multilingual-cased', 'metrics_csv': '/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/explainability/English/distil_distilbert-base-multilingual-cased/explainability_metrics_per_file.csv', 'htmls': 172}
Hindi {'out_dir': '/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/explainability/Hindi/distil_distilbert-base-multilingual-cased', 'metrics_csv': '/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/explainability/Hindi/distil_distilbert-base-multilingual-cased/explainability_metrics_per_file.csv', 'htmls': 31}
Marathi {'out_dir': '/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/explainability/Marathi/distil_distilbert-base-multilingual-cased', 'metrics_csv': '/content/d